# Clasificación de solicitudes de tarjeta

## 1. Objetivo y datos

En un departamento de admisión de crédito, aprobar una operación problemática puede generar pérdidas y rechazar una solicitud adecuada puede suponer perder negocio. Para apoyar ese análisis interesa estudiar qué información aportan las características personales, laborales y financieras del solicitante y qué errores cometen los modelos al clasificar las solicitudes.

El proyecto utiliza dos archivos de **Credit card Details**, publicados en Kaggle: uno contiene los atributos y otro la decisión asociada a cada identificador. La fuente define `label` como **0 = aprobación registrada** y **1 = rechazo registrado**. Por tanto, se intentará clasificar esa decisión histórica, con el rechazo como clase positiva. No se dispone de un impago observado ni de las fechas necesarias para demostrar una predicción anterior a la concesión.

Se compararán regresión logística, árbol de decisión y Random Forest, considerando F1, recall, precisión, ROC-AUC y los errores de clasificación. La interpretación tendrá en cuenta tanto la detección de rechazos como las falsas alertas sobre aprobaciones y la posibilidad de explicar las predicciones.

El primer paso es conocer las dos tablas, comprobar su correspondencia y entender qué representan sus campos. Después se revisará la calidad y se definirá la evaluación, antes de explorar el entrenamiento, justificar los tratamientos y comparar modelos. El [README](../README.md) amplía el contexto y las reglas de comparación; la [memoria](../docs/Memoria-proyecto.md) recoge las decisiones y resultados del recorrido.

### Importaciones

In [1]:
# Datos
import numpy as np
import pandas as pd

# Evaluación
from sklearn.model_selection import StratifiedGroupKFold

# Utilidades
import json
from hashlib import sha256
from pathlib import Path
from IPython.display import display

### Fuentes y carga

Cargamos los [dos archivos](../data/README.md): atributos y etiquetas. Comprobamos sus dimensiones antes de unirlos por `Ind_ID`.

In [2]:
RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
DATOS = RAIZ / "data" / "raw"

clientes = pd.read_csv(DATOS / "Credit_card.csv", dtype={"Ind_ID": "string"})
etiquetas = pd.read_csv(DATOS / "Credit_card_label.csv", dtype={"Ind_ID": "string"})

display(pd.DataFrame(
    [clientes.shape, etiquetas.shape],
    index=["Atributos", "Etiquetas"],
    columns=["Registros", "Columnas"],
))

,Registros,Columnas
Atributos,1548,18
Etiquetas,1548,2


### Correspondencia entre tablas

Comprobamos que las claves sean únicas y estén informadas, y unimos las tablas sin perder ni multiplicar registros.

In [3]:
for nombre, tabla in {"Atributos": clientes, "Etiquetas": etiquetas}.items():
    if tabla["Ind_ID"].isna().any() or not tabla["Ind_ID"].is_unique:
        raise ValueError(f"{nombre}: revisar identificadores ausentes o repetidos.")

datos = clientes.merge(
    etiquetas, on="Ind_ID", how="outer", validate="one_to_one", indicator=True,
)
display(datos["_merge"].value_counts().rename_axis("Correspondencia").to_frame("Registros"))

if not datos["_merge"].eq("both").all():
    raise ValueError("Hay identificadores sin correspondencia entre las fuentes.")
datos = datos.drop(columns="_merge")

,Registros
Correspondencia,
both,1548
left_only,0
right_only,0


Las **1.548 claves coinciden en ambas tablas**, sin registros perdidos ni multiplicados. Mostramos la tabla unida con los atributos y `label`, omitiendo el identificador en esta vista.

In [4]:
display(datos.drop(columns="Ind_ID").head())

,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
0,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2,1
1,F,Y,Y,2,135000.0,Working,Secondary / secondary special,Married,House / apartment,-15761.0,-3173,1,0,0,0,Laborers,4,0
2,F,N,Y,0,247500.0,Commercial associate,Higher education,Separated,Rented apartment,-17016.0,-1347,1,0,0,0,Core staff,1,0
3,M,Y,Y,0,157500.0,Working,Secondary / secondary special,Married,House / apartment,-9927.0,-828,1,0,0,0,Drivers,2,0
4,F,N,Y,2,216000.0,State servant,Higher education,Married,House / apartment,-15444.0,-3112,1,0,0,1,NaN,4,0


### Primera lectura de los registros

Revisamos tipos, ausencias y número de valores distintos para localizar campos que requieran atención. El [diccionario](../docs/Diccionario-datos.md) aclara su significado.

In [5]:
display(pd.DataFrame({
    "Tipo leído": datos.dtypes.astype(str),
    "Ausentes": datos.isna().sum(),
    "Valores distintos": datos.nunique(),
}))

,Tipo leído,Ausentes,Valores distintos
Ind_ID,string,0,1548
GENDER,str,7,2
Car_Owner,str,0,2
Propert_Owner,str,0,2
CHILDREN,int64,0,6
Annual_income,float64,23,115
Type_Income,str,0,4
EDUCATION,str,0,5
Marital_status,str,0,5
Housing_type,str,0,6


`Mobile_phone` vale `1` en todas las observaciones: no distingue solicitudes y se puede excluir de los predictores.

`Type_Occupation` tiene **488 ausencias (31,52 %)**. Antes de imputarlas, conviene comprobar si responden a situaciones laborales diferentes.

### Etiqueta y entradas candidatas

Comprobamos los valores y el reparto de `label`. Separamos los atributos en `X`, la etiqueta en `y` y el identificador para control.

In [6]:
if datos["label"].isna().any() or not datos["label"].isin([0, 1]).all():
    raise ValueError("La etiqueta debe estar informada y contener únicamente 0 o 1.")

distribucion = datos["label"].value_counts().sort_index().to_frame("Registros")
distribucion["Porcentaje"] = (100 * distribucion["Registros"] / len(datos)).round(2)
distribucion.index = distribucion.index.map({0: "0 · Aprobación", 1: "1 · Rechazo"})
display(distribucion)

X = datos.drop(columns=["Ind_ID", "label"])
y = datos["label"]
identificadores = datos["Ind_ID"]

,Registros,Porcentaje
label,,
0 · Aprobación,1373,88.7
1 · Rechazo,175,11.3


### Lectura del bloque

Las tablas se emparejan en **1.548 registros**, con **17 atributos candidatos** y una etiqueta. Los **175 rechazos (11,30 %)** muestran que la exactitud por sí sola sería insuficiente para valorar la detección de la clase positiva.

Hay ausencias en género, ingreso, nacimiento y ocupación que requieren interpretación. La clave única no prueba que todos los registros correspondan a personas distintas. La falta de fechas y de una regla documentada de aprobación limita la interpretación a decisiones históricas.

Se conservan los valores de origen. La siguiente sección revisará la calidad y definirá las particiones antes de explorar relaciones o aprender tratamientos.

## 2. Calidad y evaluación

Revisamos códigos, coherencia y perfiles repetidos antes de fijar las muestras de evaluación. Conservamos los valores originales; los tratamientos que necesiten aprender de los datos se decidirán sobre entrenamiento.

### Códigos y coherencia básica

Comprobamos los dominios binarios, los recuentos y la coherencia entre los días de nacimiento y empleo. Las alertas señalan casos que requieren interpretación; no implican una corrección automática.

In [7]:
dominios = {
    "GENDER": ["F", "M"], "Car_Owner": ["Y", "N"], "Propert_Owner": ["Y", "N"],
    "Mobile_phone": [0, 1], "Work_Phone": [0, 1], "Phone": [0, 1], "EMAIL_ID": [0, 1],
}
alertas = {
    f"{campo}: fuera de dominio": (datos[campo].notna() & ~datos[campo].isin(valores)).sum()
    for campo, valores in dominios.items()
}
alertas.update({
    "Valores infinitos en campos numéricos": np.isinf(datos.select_dtypes("number")).sum().sum(),
    "Ingreso negativo": datos["Annual_income"].lt(0).sum(),
    "Hijos negativos o fraccionarios": (datos["CHILDREN"].lt(0) | datos["CHILDREN"].mod(1).ne(0)).sum(),
    "Familia menor que 1 o fraccionaria": (datos["Family_Members"].lt(1) | datos["Family_Members"].mod(1).ne(0)).sum(),
    "Nacimiento no negativo": datos["Birthday_count"].ge(0).sum(),
    "Inicio de empleo anterior al nacimiento": (
        datos["Employed_days"].le(0) & datos["Employed_days"].lt(datos["Birthday_count"])
    ).sum(),
    "Familia menor o igual al número de hijos": datos["Family_Members"].le(datos["CHILDREN"]).sum(),
})
display(pd.Series(alertas, name="Casos").to_frame())

,Casos
GENDER: fuera de dominio,0
Car_Owner: fuera de dominio,0
Propert_Owner: fuera de dominio,0
Mobile_phone: fuera de dominio,0
Work_Phone: fuera de dominio,0
Phone: fuera de dominio,0
EMAIL_ID: fuera de dominio,0
Valores infinitos en campos numéricos,0
Ingreso negativo,0
Hijos negativos o fraccionarios,0


Los dominios y rangos comprobados no presentan incidencias, salvo **dos registros con un hijo y un miembro de familia**. Se conservan: la definición disponible no permite resolver esa discrepancia. Los extremos plausibles se estudiarán después en entrenamiento.

### Empleo y ausencia de ocupación

Los días de empleo positivos necesitan una lectura distinta de la antigüedad. Comprobamos qué código aparece y su relación con el tipo de ingreso y las ausencias de ocupación.

In [8]:
empleo_positivo = datos["Employed_days"].gt(0)
display(datos.loc[empleo_positivo, "Employed_days"].value_counts().to_frame("Registros"))

contexto_empleo = datos.assign(
    Codigo_empleo=empleo_positivo.map({True: "Positivo", False: "No positivo"}),
    Sin_ocupacion=datos["Type_Occupation"].isna(),
).groupby(["Type_Income", "Codigo_empleo"], dropna=False).agg(
    Registros=("Ind_ID", "size"),
    Sin_ocupacion=("Sin_ocupacion", "sum"),
)
display(contexto_empleo)

,Registros
Employed_days,
365243,261


Registros  Sin_ocupacion
Type_Income          Codigo_empleo                          
Commercial associate No positivo          365             77
Pensioner            No positivo            8              3
                     Positivo             261            261
State servant        No positivo          116             18
Working              No positivo          798            129

El código **365243** aparece en **261 registros**, todos con ingreso `Pensioner` y ocupación ausente: no representa una antigüedad real. Quedan **227 ausencias de ocupación** entre los demás registros. Se conserva esta distinción para preparar las variables; no se interpreta toda ausencia como desempleo ni se imputa todavía.

### Perfiles repetidos y etiquetas distintas

Buscamos coincidencias en los 17 atributos, sin usar el identificador ni la etiqueta y considerando iguales las ausencias. Un perfil idéntico no demuestra que se trate de la misma persona.

In [9]:
# El orden de la fuente fija una base reproducible para las particiones.
datos_modelo = datos.set_index("Ind_ID").loc[clientes["Ind_ID"]].copy()
columnas_perfil = clientes.columns.drop("Ind_ID").tolist()
grupos_perfil = datos_modelo.groupby(columnas_perfil, dropna=False, sort=False).ngroup()
grupos_perfil = grupos_perfil.rename("grupo_perfil")

perfiles = datos_modelo.assign(grupo_perfil=grupos_perfil).groupby("grupo_perfil").agg(
    Registros=("label", "size"), Etiquetas=("label", "nunique"),
)
repetidos = perfiles["Registros"].gt(1)
discordantes = perfiles["Etiquetas"].gt(1)
display(pd.Series({
    "Perfiles distintos": len(perfiles),
    "Perfiles repetidos": repetidos.sum(),
    "Registros en perfiles repetidos": perfiles.loc[repetidos, "Registros"].sum(),
    "Perfiles con ambas etiquetas": discordantes.sum(),
    "Registros en perfiles con ambas etiquetas": perfiles.loc[discordantes, "Registros"].sum(),
}, name="Recuento").to_frame())

,Recuento
Perfiles distintos,1377
Perfiles repetidos,150
Registros en perfiles repetidos,321
Perfiles con ambas etiquetas,9
Registros en perfiles con ambas etiquetas,22


Hay **1.377 perfiles distintos**; **150** se repiten y reúnen **321 registros**. En **9 perfiles (22 registros)** aparecen aprobación y rechazo. Se conservan todas las filas y sus etiquetas, porque no hay evidencia para decidir qué registro sería erróneo.

Cada perfil permanecerá en una sola muestra de evaluación. Esto evita evaluar un perfil presente en entrenamiento, aunque no garantiza separar personas que tengan atributos diferentes.

### Campos para modelar

Excluimos `Mobile_phone`, que es constante. La clave y el grupo quedan como controles fuera de los predictores; los otros 16 atributos siguen siendo candidatos, con su pertinencia y preparación pendientes.

In [10]:
columnas_modelo = [campo for campo in columnas_perfil if campo != "Mobile_phone"]
X = datos_modelo[columnas_modelo].copy()
y = datos_modelo["label"].copy()
identificadores = datos_modelo.index

display(pd.Series({
    "Registros conservados": len(X),
    "Entradas candidatas": X.shape[1],
    "Atributos excluidos por constancia": len(columnas_perfil) - X.shape[1],
}, name="Recuento").to_frame())

,Recuento
Registros conservados,1548
Entradas candidatas,16
Atributos excluidos por constancia,1


### Reglas de evaluación

Reservamos aproximadamente un **20 % para el cierre** y usamos **cinco folds agrupados y estratificados** en desarrollo: cada perfil se valida una vez y permanece completo. Todos los candidatos compartirán esas particiones.

La comparación principal será el **F1 medio de la clase rechazo**, con umbral inicial **0,5**; se acompañará de dispersión entre folds, recall, precisión, ROC-AUC y matriz de confusión. Las referencias serán predecir siempre la clase mayoritaria y predecir siempre rechazo.

Dentro de **0,01 de F1** del mejor resultado se priorizará la familia más sencilla: logística, árbol y bosque, en ese orden; dentro de una familia, menor dispersión y después mayor F1. Es una tolerancia práctica, no un umbral de utilidad bancaria ni una prueba de equivalencia. Cualquier ajuste posterior de umbral se hará solo en desarrollo.

### Reserva y validación por perfiles

Fijamos las semillas y el orden de entrada con la [configuración de evaluación](../evaluacion/configuracion.json). La primera exploración utilizará solo el entrenamiento del primer fold; los tratamientos se volverán a ajustar dentro de cada fold al comparar modelos.

In [11]:
EVALUACION = RAIZ / "evaluacion"
configuracion = json.loads((EVALUACION / "configuracion.json").read_text(encoding="utf-8"))
for archivo, huella in configuracion["sha256_fuentes"].items():
    if sha256((DATOS / archivo).read_bytes()).hexdigest() != huella:
        raise ValueError("Los datos han cambiado: revisar la configuración antes de repartir.")

reserva = StratifiedGroupKFold(
    n_splits=configuracion["bloques_reserva"], shuffle=True,
    random_state=configuracion["semilla_test"],
)
i_dev, i_test = next(reserva.split(X, y, groups=grupos_perfil))
X_dev, y_dev, grupos_dev = X.iloc[i_dev].copy(), y.iloc[i_dev].copy(), grupos_perfil.iloc[i_dev].copy()
X_test, y_test, grupos_test = X.iloc[i_test].copy(), y.iloc[i_test].copy(), grupos_perfil.iloc[i_test].copy()

cv = StratifiedGroupKFold(
    n_splits=configuracion["folds_desarrollo"], shuffle=True,
    random_state=configuracion["semilla_cv"],
)
particiones = list(cv.split(X_dev, y_dev, groups=grupos_dev))
i_train, i_valid = particiones[0]
X_train, y_train = X_dev.iloc[i_train].copy(), y_dev.iloc[i_train].copy()
X_valid, y_valid = X_dev.iloc[i_valid].copy(), y_dev.iloc[i_valid].copy()

In [12]:
assert X.index.equals(y.index) and X.index.equals(grupos_perfil.index)
assert len(np.unique(np.concatenate([i_dev, i_test]))) == len(X)
assert set(grupos_dev).isdisjoint(grupos_test)
assert y_dev.nunique() == y_test.nunique() == 2

resumen_folds = []
for fold, (tr, va) in enumerate(particiones, start=1):
    assert not np.intersect1d(tr, va).size
    assert len(tr) + len(va) == len(X_dev)
    assert set(grupos_dev.iloc[tr]).isdisjoint(grupos_dev.iloc[va])
    assert y_dev.iloc[tr].nunique() == y_dev.iloc[va].nunique() == 2
    resumen_folds.append({
        "Fold": fold, "Entrenamiento": len(tr), "Rechazos entrenamiento": int(y_dev.iloc[tr].sum()),
        "Validación": len(va), "Rechazos validación": int(y_dev.iloc[va].sum()),
    })
assert np.array_equal(np.sort(np.concatenate([va for _, va in particiones])), np.arange(len(X_dev)))

resumen_muestras = pd.DataFrame([
    {"Muestra": nombre, "Registros": len(etiqueta), "Perfiles": grupo.nunique(),
     "Rechazos": int(etiqueta.sum()), "Rechazos (%)": round(100 * etiqueta.mean(), 2)}
    for nombre, etiqueta, grupo in [("Desarrollo", y_dev, grupos_dev), ("Test", y_test, grupos_test)]
]).set_index("Muestra")
resumen_folds = pd.DataFrame(resumen_folds).set_index("Fold")
display(resumen_muestras, resumen_folds)

,Registros,Perfiles,Rechazos,Rechazos (%)
Muestra,,,,
Desarrollo,1238,1102,140,11.31
Test,310,275,35,11.29


,Entrenamiento,Rechazos entrenamiento,Validación,Rechazos validación
Fold,,,,
1,990,112,248,28
2,990,112,248,28
3,991,112,247,28
4,990,112,248,28
5,991,112,247,28


Quedan **1.238 registros en desarrollo y 310 en test**, con **140 y 35 rechazos**, respectivamente. Cada validación contiene **28 rechazos** y no comparte perfiles con su entrenamiento. La primera exploración dispone de **990 registros de entrenamiento**.

El conjunto público ya ha sido examinado: esta reserva permite mantener separado el cierre durante el recorrido, pero sus resultados serán una **evaluación interna**, no una prueba nueva e independiente. Además, el reducido número de rechazos aconseja interpretar con cautela diferencias pequeñas.

### Conservación de las particiones

Guardamos la asignación por identificador y fold para reutilizarla sin cambiar el reparto. El archivo individual queda excluido de Git; la configuración y los resúmenes agregados sí se pueden consultar en el repositorio.

In [13]:
registro_particiones = grupos_perfil.to_frame().assign(uso="test", orden_en_uso=0, fold_validacion=0)
registro_particiones.loc[X_dev.index, "uso"] = "desarrollo"
registro_particiones.loc[X_dev.index, "orden_en_uso"] = np.arange(len(X_dev))
registro_particiones.loc[X_test.index, "orden_en_uso"] = np.arange(len(X_test))
for fold, (_, va) in enumerate(particiones, start=1):
    registro_particiones.loc[X_dev.index[va], "fold_validacion"] = fold

ruta_particiones = EVALUACION / "particiones.csv"
if ruta_particiones.exists():
    guardadas = pd.read_csv(ruta_particiones, dtype={"Ind_ID": "string"}).set_index("Ind_ID")
    pd.testing.assert_frame_equal(guardadas.sort_index(), registro_particiones.sort_index())
else:
    registro_particiones.to_csv(ruta_particiones)
resumen_muestras.to_csv(EVALUACION / "resumen_muestras.csv")
resumen_folds.to_csv(EVALUACION / "resumen_folds.csv")
print("Particiones conservadas y comprobadas; resúmenes de muestras y folds guardados.")

Particiones conservadas y comprobadas; resúmenes de muestras y folds guardados.


### Lectura del bloque

Se conservan los **1.548 registros** y se retira únicamente el móvil constante de los predictores. Las alertas familiares, las ausencias y el código de empleo quedan identificados; no se han imputado valores, agrupado categorías ni aprendido transformaciones.

Los perfiles completos quedan separados entre desarrollo y test y entre entrenamiento y validación. Están fijados el criterio de comparación y las referencias, pero todavía no hay modelos entrenados ni métricas predictivas. El siguiente paso será explorar el entrenamiento, justificar agrupaciones con sentido y decidir la preparación de las variables.

## 3. Exploración y preparación

Pendiente de desarrollo: estudiar las variables en el entrenamiento del primer fold, justificar las agrupaciones categóricas y definir los tratamientos que se ajustarán dentro de cada partición.